In [1]:
Workspace_id ="21c5455c-dba4-427e-91ea-f700c22f242c"
lakehouse_id ="3d134afc-0b88-495d-a65f-6412b6d2ac6d"
folderpath ="Files/bronze"

sourcepath = f'abfss://{Workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/{folderpath}'

print(sourcepath)
# f'abfss://{workspace_ID}@onelake.dfs.fabric.microsoft.com/{Lakehouse_ID}/{Folder}/{customer}'




StatementMeta(, dea664d5-fcb3-4be7-ac04-2170e154da9a, 3, Finished, Available, Finished, False)

abfss://21c5455c-dba4-427e-91ea-f700c22f242c@onelake.dfs.fabric.microsoft.com/3d134afc-0b88-495d-a65f-6412b6d2ac6d/Files/bronze


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

tables = [
    "raw_customers",
    "raw_inventory",
    "raw_products",
    "raw_sales_transactions",
    "raw_stores",
]

for table in tables:
    path = f"{sourcepath}/{table}/"
    df = spark.read.parquet(path)


    # Cast _ingested_at to proper timestamp
    from pyspark.sql.functions import to_timestamp, col
    df = df.withColumn("_ingested_at", to_timestamp(col("_ingested_at")))
    
    # Re-write as Delta with correct types
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"bronze.{table}")
    
    print(f"✅ created: bronze.{table}")

print("All done!")



StatementMeta(, dea664d5-fcb3-4be7-ac04-2170e154da9a, 4, Finished, Available, Finished, False)

✅ Fixed: bronze.raw_customers
✅ Fixed: bronze.raw_inventory
✅ Fixed: bronze.raw_products
✅ Fixed: bronze.raw_sales_transactions
✅ Fixed: bronze.raw_stores
All done!
